# Immunotherapy Patient Extraction
Filters medication orders for a target list of immunotherapy drugs and writes two output sheets:
- **Orders** – one row per order
- **Summary** – one row per patient × drug, with earliest start and latest end date

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_FILE  = "your_file.csv"    # change to your input file name
OUTPUT_FILE = None               # None → auto-named  <input>_immunotherapy.xlsx

IMMUNOTHERAPY_DRUGS = [
    "Atezolizumab", "Pembrolizumab", "Durvalumab", "Nivolumab",
    "Relatlimab",   "Cemiplimab",    "Avelumab",   "Ipilimumab",
]

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.strip()
print(f"Loaded {len(df):,} rows — columns: {list(df.columns)}")

In [ ]:
# ── Filter immunotherapy orders ───────────────────────────────────────────────
pattern  = "|".join(IMMUNOTHERAPY_DRUGS)
filtered = df[df["EPIC_MEDICATION_NAME"].str.contains(pattern, case=False, na=False)].copy()

print(f"Filtered rows   : {len(filtered):,}")
print(f"Unique patients : {filtered['IP_PATIENT_ID'].nunique():,}")
filtered.head()

In [ ]:
# ── Sheet 1 – Orders (one row per order) ─────────────────────────────────────
sheet1_cols = ["IP_PATIENT_ID", "ORDER_DATE", "START_DATE", "END_DATE",
               "EPIC_MED_ID", "EPIC_MEDICATION_NAME"]
sheet1_cols = [c for c in sheet1_cols if c in filtered.columns]

sheet1 = filtered[sheet1_cols].reset_index(drop=True)
sheet1.head()

In [ ]:
# ── Sheet 2 – Summary (earliest start / latest end per patient × drug) ────────
group_cols = [c for c in ["IP_PATIENT_ID", "EPIC_MED_ID", "EPIC_MEDICATION_NAME"]
              if c in filtered.columns]

agg = {}
if "START_DATE" in filtered.columns: agg["START_DATE"] = "min"
if "END_DATE"   in filtered.columns: agg["END_DATE"]   = "max"

sheet2 = (
    filtered.groupby(group_cols, as_index=False)
    .agg(agg)
    .sort_values(["IP_PATIENT_ID", "EPIC_MEDICATION_NAME"])
    .reset_index(drop=True)
)
sheet2.head()

In [ ]:
# ── Write output ──────────────────────────────────────────────────────────────
stem = Path(INPUT_FILE).stem if OUTPUT_FILE is None else Path(OUTPUT_FILE).stem

xlsx_file    = stem + "_immunotherapy.xlsx"
csv_orders   = stem + "_immunotherapy_orders.csv"
csv_summary  = stem + "_immunotherapy_summary.csv"

with pd.ExcelWriter(xlsx_file, engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    sheet1.to_excel(writer, sheet_name="Orders",  index=False)
    sheet2.to_excel(writer, sheet_name="Summary", index=False)

sheet1.to_csv(csv_orders,  index=False)
sheet2.to_csv(csv_summary, index=False)

print(f"Excel  : {xlsx_file}")
print(f"CSV    : {csv_orders}")
print(f"CSV    : {csv_summary}")